ROM (Relational Orbital Mechanics) — algebraic verification & classification
============================================================================

Goal (per task brief):
  * Algebraically TEST the closed system (internal consistency of every
    multi-form identity), assuming the *derivations* are correct.
  * CLASSIFY every relation: is it (D) a definitional/redundant identity,
    (C) equivalent to a classical Newton/Kepler/SR result, (G) equivalent to
    a General-Relativity (Schwarzschild) result, (X) a deviation from standard
    physics, (N) a parametrisation-specific identity with no obvious textbook
    twin, or (A) an auxiliary definition that merely introduces a new symbol.

Method:
  * IDENTITY TESTS are done numerically at many random, high-precision points.
    We randomise c, G, M, a, e, and phase O.  Randomising c and G as well means
    any identity that is not dimensionally/units-consistent will FAIL — a strong
    correctness check, not just a coincidence-at-one-point check.
  * The "ground truth" assignment is the ordinary physical one:
        R_s = 2GM/c^2 ,   beta^2 = GM/(c^2 a)  (== circular speed at radius a),
        kappa^2 = R_s/a = 2 beta^2  (closure built in),
        T = 2*pi*sqrt(a^3/GM)  (Kepler III),
        r(O) = a(1-e^2)/(1+e cosO),  vis-viva for local speed, etc.
    If every ROM alternative form collapses to an identity under THIS
    assignment, then (a) the system is internally consistent AND (b) we learn
    exactly which textbook relation each form equals.
  * SERIES COMPARISONS (precession, light deflection) are done symbolically
    with sympy and compared term-by-term to the standard Schwarzschild results.


In [3]:
import mpmath as mp
import sympy as sp
import random

mp.mp.dps = 50          # 50 significant digits
TOL = mp.mpf(10) ** (-30)
random.seed(20260629)

# ----------------------------------------------------------------------
# Numerical "ground-truth" model: every quantity as a function of the
# independent physical parameters (c, G, M, a, e, O).
# ----------------------------------------------------------------------
def model(c, G, M, a, e, O):
    """Return a dict of all ROM quantities evaluated from base physics."""
    sin, cos, sqrt = mp.sin, mp.cos, mp.sqrt
    pi = mp.pi
    V = {}
    V['c'], V['G'], V['M'], V['a'], V['e'], V['O'] = c, G, M, a, e, O

    # --- base scales ---
    Rs = 2*G*M/c**2
    V['Rs'] = Rs
    beta  = sqrt(G*M/(c**2*a))          # global kinetic projection (circular speed at a)
    kappa = sqrt(Rs/a)                  # global potential projection
    V['beta'], V['kappa'] = beta, kappa
    V['betaY'] = sqrt(1-beta**2)
    V['kappaX'] = sqrt(1-kappa**2)
    V['tau']   = V['kappaX']*V['betaY']
    V['tauY']  = sqrt(1-V['tau']**2)
    V['Q']     = sqrt(kappa**2+beta**2)
    V['QY']    = sqrt(1-V['Q']**2)

    # --- orbit scale / Kepler ---
    T = 2*pi*sqrt(a**3/(G*M))
    V['T'] = T
    V['omega'] = 2*pi/T
    eY = sqrt(1-e**2); eX = (1+e)/(1-e)
    V['eY'], V['eX'] = eY, eX

    # --- densities ---
    V['rho']    = kappa**2*c**2/(8*pi*G*a**2)
    V['rhomax'] = c**2/(8*pi*G*a**2)

    # --- angular momentum (specific), textbook h = sqrt(GM a (1-e^2)) ---
    V['hW'] = sqrt(G*M*a*(1-e**2))

    # --- perihelion / aphelion ---
    V['rp'] = a*(1-e); V['ra'] = a*(1+e)
    V['kappa_p'] = kappa*sqrt(1/(1-e))
    V['kappa_a'] = sqrt(beta**2 + (beta*sqrt((1-e)/(1+e)))**2)
    V['beta_p']  = beta*sqrt((1+e)/(1-e))
    V['beta_a']  = beta*sqrt((1-e)/(1+e))
    V['kappaXp'] = sqrt(1-V['kappa_p']**2)
    V['delta_p'] = mp.mpf(1)/(1+e)
    V['delta_a'] = mp.mpf(1)/(1-e)
    V['Qp'] = sqrt(V['kappa_p']**2+V['beta_p']**2)
    V['Qa'] = sqrt(V['kappa_a']**2+V['beta_a']**2)

    # --- local phase quantities at true phase O ---
    r_o = a*(1-e**2)/(1+e*cos(O))
    V['r_o'] = r_o
    V['eta_o'] = (1-e**2)/(1+e*cos(O))
    V['kappa_o'] = sqrt(Rs/r_o)
    V['beta_T'] = beta*(1+e*cos(O))/eY
    V['beta_R'] = beta*e*sin(O)/eY
    V['beta_o'] = sqrt(V['beta_R']**2+V['beta_T']**2)
    V['betaYo'] = sqrt(1-V['beta_o']**2)
    V['kappaXo'] = sqrt(1-V['kappa_o']**2)
    V['omega_o'] = (beta*c/a)*(1+e*cos(O))**2/(1-e**2)**mp.mpf(1.5)
    V['delta_o'] = (1+e*cos(O))/(1+e**2+2*e*cos(O))
    V['Qo'] = sqrt(V['kappa_o']**2+V['beta_o']**2)
    V['tau_o'] = V['kappaXo']*V['betaYo']
    V['tauYo'] = sqrt(1-V['tau_o']**2)
    V['t_o'] = r_o/c
    V['Zsys'] = (1/V['tau'])

    # redshifts
    V['z_kappa'] = 1/V['kappaX'] - 1
    V['z_beta']  = 1/V['betaY'] - 1
    V['beta_int'] = beta/eY

    # balance point
    V['Ba'] = mp.acos(-e)
    V['Bd'] = 2*pi - mp.acos(-e)
    return V


def random_params():
    c = mp.mpf(random.uniform(0.6, 1.8))
    G = mp.mpf(random.uniform(0.6, 1.8))
    M = mp.mpf(random.uniform(0.5, 2.0))
    a = mp.mpf(random.uniform(80.0, 400.0))   # a >> R_s : weak field, kappa<1
    e = mp.mpf(random.uniform(0.05, 0.7))
    O = mp.mpf(random.uniform(0.2, 2*float(mp.pi)-0.2))
    return c, G, M, a, e, O


# ----------------------------------------------------------------------
# Identity test harness
# ----------------------------------------------------------------------
RESULTS = []   # (section, label, category, passed, detail)

def check(label, category, lhs, rhs, section, npts=12, fixedO=None):
    """lhs, rhs are callables V -> mpmath value. Test equality over random pts."""
    worst = mp.mpf(0)
    ok = True
    err = None
    for _ in range(npts):
        c, G, M, a, e, O = random_params()
        if fixedO is not None:
            O = mp.mpf(fixedO)
        V = model(c, G, M, a, e, O)
        try:
            L = lhs(V); R = rhs(V)
        except Exception as ex:
            ok = False; err = ex; break
        denom = max(abs(L), abs(R), mp.mpf(1))
        rel = abs(L-R)/denom
        if rel > worst:
            worst = rel
        if rel > TOL:
            ok = False
    RESULTS.append((section, label, category, ok, (err if err is not None else worst)))
    return ok


# ======================================================================
print("="*78)
print("ROM ALGEBRAIC VERIFICATION  (50-digit, random c,G,M,a,e,O)")
print("="*78)

# ---------------------------------------------------------------- closure
S = "1. Closure law & global projections"
check("kappa^2 = 2 beta^2  (CLOSURE)", "C: Newtonian circular-orbit v^2=GM/a",
      lambda V: V['kappa']**2, lambda V: 2*V['beta']**2, S)
check("beta^2 = GM/(c^2 a)  (circular speed at a)", "C",
      lambda V: V['beta']**2, lambda V: V['G']*V['M']/(V['c']**2*V['a']), S)
check("kappa = sqrt(Rs/a)", "D",
      lambda V: V['kappa'], lambda V: mp.sqrt(V['Rs']/V['a']), S)
check("Q = sqrt(3) beta = sqrt(3/2) kappa", "D: repackaging",
      lambda V: V['Q'], lambda V: mp.sqrt(3)*V['beta'], S)
check("Q^2 = kappa^2 + beta^2", "D",
      lambda V: V['Q']**2, lambda V: V['kappa']**2+V['beta']**2, S)

# ---------------------------------------------------------------- redshift / SR-GR factors
S = "2. Redshift & phase factors"
check("kappa^2 = 1-1/(1+z_kappa)^2  (grav redshift)", "G: Schwarzschild sqrt(1-Rs/r)",
      lambda V: V['kappa']**2, lambda V: 1-1/(1+V['z_kappa'])**2, S)
check("beta^2 = 1-1/(1+z_beta)^2  (transverse Doppler)", "C: SR gamma factor",
      lambda V: V['beta']**2, lambda V: 1-1/(1+V['z_beta'])**2, S)
check("kappaX = 1/(1+z_kappa) = sqrt(1-kappa^2)", "G: Schwarzschild time-dilation",
      lambda V: V['kappaX'], lambda V: 1/(1+V['z_kappa']), S)
check("betaY = 1/(1+z_beta) = 1/gamma", "C: SR",
      lambda V: V['betaY'], lambda V: 1/(1+V['z_beta']), S)
check("Zsys = (1+z_kappa)(1+z_beta) = 1/tau", "C+G: multiplicative redshift combination",
      lambda V: (1+V['z_kappa'])*(1+V['z_beta']), lambda V: 1/V['tau'], S)
check("tau = kappaX*betaY", "D",
      lambda V: V['tau'], lambda V: V['kappaX']*V['betaY'], S)
check("tauY^2 = 3 beta^2 - 2 beta^4", "N: combined-dilation defect (drives precession)",
      lambda V: V['tauY']**2, lambda V: 3*V['beta']**2-2*V['beta']**4, S)

# ---------------------------------------------------------------- Kepler / scales
S = "3. Orbital scales (Kepler III etc.)"
check("beta = 2*pi*a/(T c)", "C: v = circumference/period",
      lambda V: V['beta'], lambda V: 2*mp.pi*V['a']/(V['T']*V['c']), S)
check("beta^3 = pi Rs/(T c)", "C: Kepler III",
      lambda V: V['beta']**3, lambda V: mp.pi*V['Rs']/(V['T']*V['c']), S)
check("Rs = 8 pi^2 a^3/(T^2 c^2)", "C: Kepler III",
      lambda V: V['Rs'], lambda V: 8*mp.pi**2*V['a']**3/(V['T']**2*V['c']**2), S)
check("a = Rs/kappa^2", "D",
      lambda V: V['a'], lambda V: V['Rs']/V['kappa']**2, S)
check("a = T beta c/(2 pi)", "C",
      lambda V: V['a'], lambda V: V['T']*V['beta']*V['c']/(2*mp.pi), S)
check("M = T c^3 beta^3/(2 pi G)", "C: Kepler III",
      lambda V: V['M'], lambda V: V['T']*V['c']**3*V['beta']**3/(2*mp.pi*V['G']), S)
check("M = kappa^2 c^2 a/(2 G)", "D: = Rs c^2/2G",
      lambda V: V['M'], lambda V: V['kappa']**2*V['c']**2*V['a']/(2*V['G']), S)
check("M = 4 pi rho a^3", "A/D: defines rho",
      lambda V: V['M'], lambda V: 4*mp.pi*V['rho']*V['a']**3, S)
check("omega = beta c/a = (c/Rs) 2 beta^3", "C",
      lambda V: V['beta']*V['c']/V['a'], lambda V: (V['c']/V['Rs'])*2*V['beta']**3, S)
check("T = 2 pi sqrt(2) Rs/(kappa^3 c)", "C: Kepler III in kappa form",
      lambda V: V['T'], lambda V: 2*mp.pi*mp.sqrt(2)*V['Rs']/(V['kappa']**3*V['c']), S)
check("rho_max = c^2/(8 pi G a^2)", "A: defines density bound",
      lambda V: V['rhomax'], lambda V: V['c']**2/(8*mp.pi*V['G']*V['a']**2), S)
check("rho = kappa^2 rho_max", "A/D",
      lambda V: V['rho'], lambda V: V['kappa']**2*V['rhomax'], S)
check("binding: beta^2 = Rs/(2a)", "C: specific orbital energy",
      lambda V: V['beta']**2, lambda V: V['Rs']/(2*V['a']), S)

# ---------------------------------------------------------------- angular momentum
S = "4. Angular momentum"
check("hW = a beta c eY  (= sqrt(GM a (1-e^2)))", "C: Kepler ang. momentum",
      lambda V: V['hW'], lambda V: V['a']*V['beta']*V['c']*V['eY'], S)
check("hW = Rs c sqrt(1-e^2)/(2 beta)", "C",
      lambda V: V['hW'], lambda V: V['Rs']*V['c']*mp.sqrt(1-V['e']**2)/(2*V['beta']), S)
check("hW = r_o^2 omega_o  (per-phase invariant)", "C: dA/dt const (Kepler II)",
      lambda V: V['hW'], lambda V: V['r_o']**2*V['omega_o'], S)

# ---------------------------------------------------------------- vis-viva / local
S = "5. Local phase kinematics (vis-viva, velocity decomposition)"
check("beta_T = r_o omega_o /c  (transverse speed)", "C",
      lambda V: V['beta_T'], lambda V: V['r_o']*V['omega_o']/V['c'], S)
check("beta_o = beta sqrt(1+e^2+2e cosO)/eY  (speed)", "C: Kepler",
      lambda V: V['beta_o'], lambda V: V['beta']*mp.sqrt(1+V['e']**2+2*V['e']*mp.cos(V['O']))/V['eY'], S)
check("beta_o^2 = Rs/r_o - Rs/(2a)  (VIS-VIVA)", "C: vis-viva v^2=GM(2/r-1/a)",
      lambda V: V['beta_o']**2, lambda V: V['Rs']/V['r_o']-V['Rs']/(2*V['a']), S)
check("kappa_o^2 = beta^2 + beta_o^2", "C: vis-viva rearranged",
      lambda V: V['kappa_o']**2, lambda V: V['beta']**2+V['beta_o']**2, S)
check("kappa_o = sqrt(Rs/r_o)", "G: local potential",
      lambda V: V['kappa_o'], lambda V: mp.sqrt(V['Rs']/V['r_o']), S)
check("r_o = Rs/kappa_o^2", "C: ellipse / D",
      lambda V: V['r_o'], lambda V: V['Rs']/V['kappa_o']**2, S)
check("eta_o = r_o/a = 2 - 2 beta_o^2/kappa_o^2", "C: vis-viva form",
      lambda V: V['eta_o'], lambda V: 2-2*V['beta_o']**2/V['kappa_o']**2, S)
check("omega_o = a beta c eY / r_o^2", "C: Kepler II",
      lambda V: V['omega_o'], lambda V: V['a']*V['beta']*V['c']*V['eY']/V['r_o']**2, S)
check("delta_o = kappa_o^2/(2 beta_o^2)", "D",
      lambda V: V['delta_o'], lambda V: V['kappa_o']**2/(2*V['beta_o']**2), S)
check("beta_T/kappa_o^2 = eY/(2 beta)  (invariant)", "C: ang.mom. invariant",
      lambda V: V['beta_T']/V['kappa_o']**2, lambda V: V['eY']/(2*V['beta']), S)
check("beta_T^2 eta_o^2 = beta^2 (1-e^2)  (invariant)", "C",
      lambda V: V['beta_T']**2*V['eta_o']**2, lambda V: V['beta']**2*(1-V['e']**2), S)

# ---------------------------------------------------------------- key invariants
S = "6. Named orbital invariants"
check("ENERGY INVARIANT: kappa_o^2 - beta_o^2 = beta^2  (const)", "C: total energy const",
      lambda V: V['kappa_o']**2-V['beta_o']**2, lambda V: V['beta']**2, S)
check("beta^2 - beta_o^2 = kappa^2 - kappa_o^2", "D: from closure+vis-viva",
      lambda V: V['beta']**2-V['beta_o']**2, lambda V: V['kappa']**2-V['kappa_o']**2, S)

# ---------------------------------------------------------------- eccentricity
S = "7. Eccentricity relations"
check("e = 2 beta_p^2/kappa_p^2 - 1", "C: from peri values",
      lambda V: V['e'], lambda V: 2*V['beta_p']**2/V['kappa_p']**2-1, S)
check("e = 1 - 2 beta_a^2/kappa_a^2", "C",
      lambda V: V['e'], lambda V: 1-2*V['beta_a']**2/V['kappa_a']**2, S)
check("eX = (1+e)/(1-e) = ra/rp", "C: ellipse geometry",
      lambda V: V['eX'], lambda V: V['ra']/V['rp'], S)
check("eX = beta_p/beta_a", "C: ang.mom. conservation",
      lambda V: V['eX'], lambda V: V['beta_p']/V['beta_a'], S)
check("eX = kappa_p^2/kappa_a^2", "C",
      lambda V: V['eX'], lambda V: V['kappa_p']**2/V['kappa_a']**2, S)
check("eX = delta_a/delta_p", "D",
      lambda V: V['eX'], lambda V: V['delta_a']/V['delta_p'], S)
check("eX = (kappa_a^2 beta_p^2)/(kappa_p^2 beta_a^2)", "C",
      lambda V: V['eX'], lambda V: (V['kappa_a']**2*V['beta_p']**2)/(V['kappa_p']**2*V['beta_a']**2), S)

# ---------------------------------------------------------------- peri/aphelion
S = "8. Perihelion / aphelion"
check("rp = a(1-e) = Rs/kappa_p^2", "C/D", lambda V: V['rp'], lambda V: V['Rs']/V['kappa_p']**2, S)
check("ra = a(1+e) = Rs/kappa_a^2", "C/D", lambda V: V['ra'], lambda V: V['Rs']/V['kappa_a']**2, S)
check("kappa_p = kappa sqrt(1/(1-e))", "C", lambda V: V['kappa_p'], lambda V: V['kappa']*mp.sqrt(1/(1-V['e'])), S)
check("beta_p = beta sqrt((1+e)/(1-e))", "C", lambda V: V['beta_p'], lambda V: V['beta']*mp.sqrt((1+V['e'])/(1-V['e'])), S)
check("beta_p = (kappa_p/sqrt2) sqrt(1+e)", "D", lambda V: V['beta_p'], lambda V: (V['kappa_p']/mp.sqrt(2))*mp.sqrt(1+V['e']), S)
check("delta_p = kappa_p^2/(2 beta_p^2) = 1/(1+e)", "C/D", lambda V: V['delta_p'], lambda V: V['kappa_p']**2/(2*V['beta_p']**2), S)
check("beta_a = beta/sqrt(eX)", "C", lambda V: V['beta_a'], lambda V: V['beta']/mp.sqrt(V['eX']), S)
check("kappa_a = sqrt(beta^2+beta_a^2)", "C", lambda V: V['kappa_a'], lambda V: mp.sqrt(V['beta']**2+V['beta_a']**2), S)
check("delta_a = kappa_a^2/(2 beta_a^2) = 1/(1-e)", "C/D", lambda V: V['delta_a'], lambda V: V['kappa_a']**2/(2*V['beta_a']**2), S)

# ---------------------------------------------------------------- balance point
S = "9. Balance points"
def _r_at(V, ang):
    return V['a']*(1-V['e']**2)/(1+V['e']*mp.cos(ang))
check("Ba = arccos(-e): at Ba, r = a", "N: ROM construct (true anomaly where r=a)",
      lambda V: _r_at(V, V['Ba']), lambda V: V['a'], S)
def _kappao2_at(V, ang):
    r = _r_at(V, ang); return V['Rs']/r
def _betao2_at(V, ang):
    r = _r_at(V, ang); return V['Rs']/r - V['Rs']/(2*V['a'])
check("at Ba: kappa_o^2 = 2 beta_o^2 (local closure)", "N",
      lambda V: _kappao2_at(V, V['Ba']), lambda V: 2*_betao2_at(V, V['Ba']), S)
check("Ba = 2pi - Bd", "D",
      lambda V: V['Ba'], lambda V: 2*mp.pi-V['Bd'], S)

# ---------------------------------------------------------------- metric translation
S = "10. Legacy tensor translation (Schwarzschild)"
check("g_tt = -kappaX^2 c^2 = -(1-Rs/r) c^2", "G: Schwarzschild g_tt EXACT",
      lambda V: -V['kappaX']**2*V['c']**2, lambda V: -(1-V['Rs']/V['a'])*V['c']**2, S)
check("g_rr = kappaX^-2 = (1-Rs/r)^-1", "G: Schwarzschild g_rr EXACT",
      lambda V: 1/V['kappaX']**2, lambda V: 1/(1-V['Rs']/V['a']), S)
check("vacuum: d/dr(r kappa^2)=0  <=> r kappa^2 = Rs (const)", "G: Schwarzschild vacuum",
      lambda V: V['kappa']**2*V['a'], lambda V: V['Rs'], S)
check("T^0_0 = kappa^2 c^4/(8 pi G r^2) = rho_max kappa^2 c^2", "X: nonzero exterior 'density' (not vacuum GR)",
      lambda V: V['kappa']**2*V['c']**4/(8*mp.pi*V['G']*V['a']**2), lambda V: V['rhomax']*V['kappa']**2*V['c']**2, S)

# ---------------------------------------------------------------- Holographic invariant
S = "11. Holographic Decryption Invariant"
def holo_lhs(V):
    # Z_raw(O)*tau(O) = 1 + beta_int*(cos(O+wi)+e cos wi) sin i ; at O=-wi and O=pi-wi
    e = V['e']; bint = V['beta']/V['eY']
    wi = mp.mpf(random.uniform(0.1, 1.2))
    inc = mp.mpf(random.uniform(0.2, 1.4))
    A = bint*mp.sin(inc)
    c0 = e*mp.cos(wi)
    term1 = (1 + A*(1+c0))*(1 - c0)     # O = -wi  -> cos(O+wi)=+1, weight (1-e cos wi)
    term2 = (1 + A*(-1+c0))*(1 + c0)    # O = pi-wi-> cos(O+wi)=-1, weight (1+e cos wi)
    return term1+term2
check("Z_raw.tau.(1-e c) + Z_raw.tau.(1+e c) = 2", "N: exact identity (conjugate-phase symmetry)",
      holo_lhs, lambda V: mp.mpf(2), S)

# ---------------------------------------------------------------- Energy symmetry
S = "12. Energy Symmetry Law"
def dE_sum(V):
    xA = mp.mpf(random.uniform(0.3,0.99)); yA = mp.mpf(random.uniform(0.3,0.99))
    xB = mp.mpf(random.uniform(0.3,0.99)); yB = mp.mpf(random.uniform(0.3,0.99))
    E0 = mp.mpf(random.uniform(0.5,2))
    dab = E0*(xB/yB - xA/yA)
    dba = E0*(xA/yA - xB/yB)
    return dab+dba
check("dE(A->B) + dE(B->A) = 0  (as defined)", "D: tautology (antisymmetric by definition)",
      dE_sum, lambda V: mp.mpf(0), S)

print()
# ----------------------------------------------------------------------
# Report identity results
# ----------------------------------------------------------------------
cur = None
npass = nfail = 0
for section, label, cat, ok, worst in RESULTS:
    if section != cur:
        cur = section
        print("\n"+section)
        print("-"*len(section))
    tag = "PASS" if ok else "**FAIL**"
    if ok: npass += 1
    else: nfail += 1
    ws = (f"{float(worst):.1e}" if not isinstance(worst, Exception) else f"ERR:{worst}")
    print(f"  [{tag}] ({cat})")
    print(f"         {label}   (max rel.dev {ws})")

print("\n" + "="*78)
print(f"IDENTITY TESTS: {npass} passed, {nfail} failed, out of {npass+nfail}")
print("="*78)

ROM ALGEBRAIC VERIFICATION  (50-digit, random c,G,M,a,e,O)


1. Closure law & global projections
-----------------------------------
  [PASS] (C: Newtonian circular-orbit v^2=GM/a)
         kappa^2 = 2 beta^2  (CLOSURE)   (max rel.dev 8.4e-53)
  [PASS] (C)
         beta^2 = GM/(c^2 a)  (circular speed at a)   (max rel.dev 4.2e-53)
  [PASS] (D)
         kappa = sqrt(Rs/a)   (max rel.dev 0.0e+00)
  [PASS] (D: repackaging)
         Q = sqrt(3) beta = sqrt(3/2) kappa   (max rel.dev 3.3e-52)
  [PASS] (D)
         Q^2 = kappa^2 + beta^2   (max rel.dev 4.2e-53)

2. Redshift & phase factors
---------------------------
  [PASS] (G: Schwarzschild sqrt(1-Rs/r))
         kappa^2 = 1-1/(1+z_kappa)^2  (grav redshift)   (max rel.dev 4.3e-51)
  [PASS] (C: SR gamma factor)
         beta^2 = 1-1/(1+z_beta)^2  (transverse Doppler)   (max rel.dev 3.2e-51)
  [PASS] (G: Schwarzschild time-dilation)
         kappaX = 1/(1+z_kappa) = sqrt(1-kappa^2)   (max rel.dev 1.3e-51)
  [PASS] (C: SR)
         betaY = 1/

---

ROM vs General Relativity: series comparison of the two CURVED-SPACETIME tests
==============================================================================

ROM provides closed forms for
  * perihelion precession:   d(\phi) = 2*\pi*\tau_Y^2/(1-e^2),  \tau_Y^2 = 3 b^2 - 2 b^4,
                             b^2 = \beta^2 = R_s/(2a)
                             => d(\phi)_ROM = \pi*(3x - x^2)/(1-e^2),  x = R_s/a
  * light deflection:        d(\gamma) = 2*arcsin( \kappa_p^2 / (1-\kappa_p^2) ),
                             kappa_p^2 = R_s/r_0  (closest approach r_0)
                             => d(\gamma)_ROM = 2*arcsin( s/(1-s) ),   s = R_s/r_0

We compute the EXACT Schwarzschild geodesic results by high-precision numerical
integration (no approximation), then extract their Taylor coefficients in the
SAME small parameter and compare term-by-term to ROM.  Geometric units G=c=1,
central mass M=1 (so R_s = 2). This is a theory-vs-theory comparison.

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ROM vs General Relativity: term-by-term series comparison of the two
classic CURVED-SPACETIME tests (perihelion precession & light deflection).

ROM closed forms:
  precession : d(phi)   = 2*pi*tau_Y^2/(1-e^2),  tau_Y^2 = 3 b^2 - 2 b^4,
               b^2 = beta^2 = R_s/(2a)   =>   d(phi) = pi*(3x - x^2)/(1-e^2), x=R_s/a
  deflection : d(gamma) = 2*arcsin( kappa_p^2/(1-kappa_p^2) ), kappa_p^2 = R_s/r_0
               => d(gamma) = 2*arcsin( s/(1-s) ),  s = R_s/r_0

We integrate the EXACT Schwarzschild geodesic numerically (50-digit), extract
Taylor coefficients in the same small parameter, and compare to ROM.
Geometric units G=c=1, M=1 (R_s=2). Theory-vs-theory (no observational data).
Run: python3 rom_gr_series.py
"""
import mpmath as mp
mp.mp.dps = 40
MASS = mp.mpf(1); Rs = 2*MASS

def gr_precession(a, e):
    """Exact Schwarzschild perihelion advance per orbit (rad)."""
    rp = a*(1-e); ra = a*(1+e); up = 1/rp; ua = 1/ra
    # turning-point conditions -> 2M/L^2 = (up+ua) - 2M(up^2+up ua+ua^2)
    L2 = 2*MASS / ((up+ua) - 2*MASS*(up**2 + up*ua + ua**2))
    u3 = 1/(2*MASS) - up - ua                     # 3rd root, sum of roots = 1/(2M)
    def integ(psi):                               # smooth psi-substitution
        u = ua + (up-ua)*mp.sin(psi)**2
        return 2/mp.sqrt(2*MASS*(u3-u))
    return 2*mp.quad(integ, [0, mp.pi/2]) - 2*mp.pi

def gr_deflection(r0):
    """Exact Schwarzschild photon deflection (rad), closest approach r0."""
    u0 = 1/r0; inv_b2 = u0**2 - 2*MASS*u0**3
    def integ(th):
        u = u0*mp.sin(th)
        return u0*mp.cos(th)/mp.sqrt(inv_b2 - u**2 + 2*MASS*u**3)
    return 2*mp.quad(integ, [0, mp.pi/2]) - mp.pi

def fit_coeffs(func, ps):
    """Solve for c1..cn in f(p)=c1 p + c2 p^2 + ... at the sample points ps."""
    n = len(ps); A = mp.matrix(n, n); b = mp.matrix(n, 1)
    for i, p in enumerate(ps):
        for j in range(n):
            A[i, j] = p**(j+1)
        b[i] = mp.re(func(p))
    x = mp.lu_solve(A, b)
    return [x[k] for k in range(n)]

print("="*78)
print("ROM vs GR  --  PERIHELION PRECESSION  (per orbit, radians)")
print("="*78)
for e in [mp.mpf('0.0'), mp.mpf('0.2'), mp.mpf('0.5')]:
    xs = [mp.mpf('0.0005')*k for k in (1, 2, 3, 4, 5, 6)]
    c = fit_coeffs(lambda x, e=e: gr_precession(Rs/x, e), xs)
    inv = 1/(1-e**2)
    print("\n e = %s" % float(e))
    print("   GR  exact : d(phi) = %s*x + %s*x^2 + %s*x^3 + ..."
          % (mp.nstr(c[0], 10), mp.nstr(c[1], 8), mp.nstr(c[2], 6)))
    print("   ROM       : d(phi) = %s*x + %s*x^2     (= pi(3x - x^2)/(1-e^2))"
          % (mp.nstr(3*mp.pi*inv, 10), mp.nstr(-mp.pi*inv, 8)))
    print("   x^1 leading : GR %s  vs  ROM 3pi/(1-e^2)=%s   match: %s"
          % (mp.nstr(c[0], 11), mp.nstr(3*mp.pi*inv, 11), abs(c[0]-3*mp.pi*inv) < mp.mpf('1e-9')))
    print("   x^2         : GR %s (POSITIVE)  vs  ROM %s (NEGATIVE)  => DIFFERENT: %s"
          % (mp.nstr(c[1], 10), mp.nstr(-mp.pi*inv, 10), abs(c[1]+mp.pi*inv) > mp.mpf('1e-3')))
    if e == 0:
        print("        (GR x^2 coeff at e=0 = 27*pi/4 = %s, a known closed-form result)"
              % mp.nstr(27*mp.pi/4, 10))

print("\n" + "="*78)
print("ROM vs GR  --  LIGHT DEFLECTION  (radians),  s = R_s/r_0")
print("="*78)
ss = [mp.mpf('0.0005')*k for k in (1, 2, 3, 4, 5, 6)]
cd = fit_coeffs(lambda s: gr_deflection(Rs/s), ss)
cr = fit_coeffs(lambda s: 2*mp.asin(s/(1-s)), ss)
print("\n   GR  exact : d(gamma) = %s*s + %s*s^2 + %s*s^3 + ..."
      % (mp.nstr(cd[0], 10), mp.nstr(cd[1], 8), mp.nstr(cd[2], 6)))
print("   ROM 2asin : d(gamma) = %s*s + %s*s^2 + %s*s^3 + ..."
      % (mp.nstr(cr[0], 10), mp.nstr(cr[1], 8), mp.nstr(cr[2], 6)))
print("   Newtonian : d(gamma) = 1.0*s    (HALF of the GR/ROM leading term)")
print("   s^1 leading : GR %s  vs  ROM %s  match: %s  (=2 = Einstein value = 2x Newton)"
      % (mp.nstr(cd[0], 11), mp.nstr(cr[0], 11), abs(cd[0]-cr[0]) < mp.mpf('1e-9')))
print("   s^2         : GR %s  vs  ROM %s  match: %s  (GR = 15pi/16-1 = %s)"
      % (mp.nstr(cd[1], 10), mp.nstr(cr[1], 10), abs(cd[1]-cr[1]) < mp.mpf('1e-3'), mp.nstr(15*mp.pi/16-1, 8)))
print("   s^3         : GR %s  vs  ROM %s  match: %s"
      % (mp.nstr(cd[2], 10), mp.nstr(cr[2], 10), abs(cd[2]-cr[2]) < mp.mpf('1e-3')))

print("\n" + "="*78)
print("STRONG-FIELD BEHAVIOUR of the ROM precession driver tau_Y^2 = 3b^2 - 2b^4")
print("="*78)
print("  Peaks at b^2 = 3/4 (kappa^2 = 3/2); returns to 0 at b^2 = 3/2 (kappa^2 = 3).")
print("  => ROM precession-per-radian is BOUNDED and non-monotonic in field strength,")
print("     whereas GR precession DIVERGES near the ISCO/photon sphere.")
print("  => The two theories coincide only at leading (weak-field) order.")

print("\n" + "="*78)
print("SUPPLEMENT A:  massive-particle 'gravitational deflection' leading order")
print("="*78)
for beta in [mp.mpf('0.3'), mp.mpf('0.7')]:
    s = mp.mpf('1e-6'); r0 = Rs/s; kp2 = Rs/r0; bp2 = beta**2
    rom = 2*mp.asin(kp2*(1+bp2)/(2*bp2 - kp2*(1+bp2)))
    std = (Rs/r0)*(1+bp2)/bp2                      # weak-field deflection of speed-beta particle
    print("   beta=%s : ROM_lead=%s   (R_s/r0)(1+b^2)/b^2=%s   match: %s"
          % (float(beta), mp.nstr(rom, 8), mp.nstr(std, 8), abs(rom-std)/std < mp.mpf('1e-5')))
print("   (beta->1 -> 2 R_s/r0 = Einstein; beta->0 -> Newtonian hyperbola)")

print("\n" + "="*78)
print("SUPPLEMENT B:  critical radii via the static map r = R_s/kappa^2")
print("="*78)
for name, k2, rk in [("ISCO", mp.mpf(1)/3, "3 R_s"),
                     ("Photon sphere", mp.mpf(2)/3, "1.5 R_s"),
                     ("Horizon", mp.mpf(1), "1 R_s")]:
    print("   %-14s kappa^2=%s -> r = %s R_s   (known GR value: %s)"
          % (name, mp.nstr(k2, 4), mp.nstr((Rs/k2)/Rs, 4), rk))
print("   All reproduce the standard Schwarzschild critical radii.")

ROM vs GR  --  PERIHELION PRECESSION  (per orbit, radians)

 e = 0.0
   GR  exact : d(phi) = 9.424777961*x + 21.20575*x^2 + 53.0144*x^3 + ...
   ROM       : d(phi) = 9.424777961*x + -3.1415927*x^2     (= pi(3x - x^2)/(1-e^2))
   x^1 leading : GR 9.4247779608  vs  ROM 3pi/(1-e^2)=9.4247779608   match: True
   x^2         : GR 21.20575041 (POSITIVE)  vs  ROM -3.141592654 (NEGATIVE)  => DIFFERENT: True
        (GR x^2 coeff at e=0 = 27*pi/4 = 21.20575041, a known closed-form result)

 e = 0.2
   GR  exact : d(phi) = 9.817477042*x + 23.060845*x^2 + 60.3206*x^3 + ...
   ROM       : d(phi) = 9.817477042*x + -3.2724923*x^2     (= pi(3x - x^2)/(1-e^2))
   x^1 leading : GR 9.8174770425  vs  ROM 3pi/(1-e^2)=9.8174770425   match: True
   x^2         : GR 23.06084451 (POSITIVE)  vs  ROM -3.272492347 (NEGATIVE)  => DIFFERENT: True

 e = 0.5
   GR  exact : d(phi) = 12.56637061*x + 38.222711*x^2 + 130.9*x^3 + ...
   ROM       : d(phi) = 12.56637061*x + -4.1887902*x^2     (= pi(3x - x^2)/(1-e^2))
   x

---


R.O.M.  —  MASS / G - FREE CLOSURE DEMONSTRATION
================================================

Claim under test (structural, NOT "ROM vs GR correctness"):
  The R.O.M. closed algebraic system determines the full orbital geometry from
  OBSERVABLE inputs alone.  Newton's constant G and the mass M are never needed
  as inputs; they appear only at the very end as optional unit conversions
  (a length R_s -> a mass in kg).  No metric, curvature, geodesic, action
  principle, perturbation series, or differential equation is used.

How the demonstration is kept honest:
  1. A "truth" system is built from (M, a, e) in SI units and its observables
     are generated with STANDARD physics, independent of ROM:
        - Keplerian ellipse           $r(O) = a(1-e^2)/(1+e \cos(O))$
        - vis-viva                    $v^2  = GM(2/r - 1/a)$
        - Kepler third law            $T    = 2*\pi*\sqrt{a^3/GM}$
        - GR gravitational redshift   $1+z_k = 1/\sqrt{1-R_s/r}$
        - SR transverse Doppler       $1+z_b = 1/\sqrt{1-v^2/c^2}$
  2. Only OBSERVABLES are handed to the ROM closure routines:
        beta (transverse-Doppler projection), e, period T,  and/or  z_kappa,z_beta.
     The closure routines have NO G and NO M in scope (enforced below).
  3. ROM reconstructs a, R_s, r_p, r_a and the whole phase-resolved orbit, and
     recovers GM (a length^3/time^2 combination, no G) and finally M (kg, G used
     ONLY here as a conversion).  Everything is checked against the truth.

In [1]:

import mpmath as mp
mp.mp.dps = 30
pi = mp.pi
sqrt, sin, cos = mp.sqrt, mp.sin, mp.cos

# ======================================================================
# 0.  Physical constants  (SI)  — used ONLY by the truth generator and by
#     the final optional "kg conversion", never inside ROM closure.
# ======================================================================
G    = mp.mpf('6.67430e-11')      # m^3 kg^-1 s^-2
c    = mp.mpf('2.99792458e8')     # m/s
Msun = mp.mpf('1.98892e30')       # kg

# ======================================================================
# 1.  TRUTH  (standard physics).  Fundamental truth = (M_true, a_true, e).
# ======================================================================
M_true   = mp.mpf('1.4')*Msun
beta_aim = mp.mpf('0.01')                          # target equilibrium beta
a_true   = G*M_true/(c**2*beta_aim**2)             # gives beta_glob = 0.01
e        = mp.mpf('0.88')

Rs_true   = 2*G*M_true/c**2
beta_glob = sqrt(G*M_true/(c**2*a_true))           # global kinetic projection
kappa_tru = sqrt(Rs_true/a_true)
T_true    = 2*pi*sqrt(a_true**3/(G*M_true))        # Kepler III
rp_true   = a_true*(1-e)
ra_true   = a_true*(1+e)

# global-projection redshifts (what the closure's redshift-mode consumes)
kappaX_t = sqrt(1-kappa_tru**2)
betaY_t  = sqrt(1-beta_glob**2)
z_kappa_glob = 1/kappaX_t - 1
z_beta_glob  = 1/betaY_t  - 1

def truth_phase(O):
    """Independent standard-physics state at true anomaly O."""
    r  = a_true*(1-e**2)/(1+e*cos(O))
    v2 = G*M_true*(2/r - 1/a_true)                 # vis-viva
    h  = sqrt(G*M_true*a_true*(1-e**2))            # specific ang. momentum
    vT = h/r
    vR = sqrt(v2 - vT**2) * (1 if sin(O) >= 0 else -1)
    kappa_o = sqrt(Rs_true/r)
    beta_o  = sqrt(v2)/c
    zk = 1/sqrt(1-kappa_o**2) - 1
    zb = 1/sqrt(1-beta_o**2)  - 1
    return dict(r=r, beta_o=beta_o, beta_T=vT/c, beta_R=vR/c,
                kappa_o=kappa_o, z_kappa=zk, z_beta=zb,
                Zsys=(1+zk)*(1+zb))

# OBSERVABLES exposed to ROM (NOTHING about M, G, a, R_s is exposed):
OBS = dict(beta=beta_glob, e=e, T=T_true,
           z_kappa=z_kappa_glob, z_beta=z_beta_glob)

# ======================================================================
# 2.  ROM CLOSURE  — functions take observables + c only.  No G, no M.
# ======================================================================
def rom_from_kinematics(beta, e, T, c):
    """Inputs: beta (dimensionless), e (dimensionless), T (s), c (m/s)."""
    kappa = sqrt(2)*beta                      # closure  kappa^2 = 2 beta^2
    a     = T*beta*c/(2*pi)                    # <-- scale WITHOUT GM
    Rs    = kappa**2 * a                       # = 2 beta^2 a   (a length)
    out = dict(beta=beta, kappa=kappa, a=a, Rs=Rs,
               rp=a*(1-e), ra=a*(1+e),
               kappaX=sqrt(1-kappa**2), betaY=sqrt(1-beta**2))
    out['tau']  = out['kappaX']*out['betaY']
    out['Zsys'] = 1/out['tau']
    out['Q']    = sqrt(kappa**2 + beta**2)
    out['GM']   = beta**2 * c**2 * a           # G*M recovered as a COMBINATION (no G)
    return out

def rom_from_redshifts(z_kappa, z_beta, e, T, c):
    """Inputs: two redshifts + e + T + c.  No G, no M."""
    beta  = sqrt(1 - 1/(1+z_beta)**2)          # transverse Doppler  -> beta
    kappa = sqrt(1 - 1/(1+z_kappa)**2)         # grav. redshift      -> kappa
    closure_residual = kappa**2 - 2*beta**2    # DATA consistency test (expect 0)
    a   = T*beta*c/(2*pi)
    Rs_from_kappa = kappa**2 * a
    Rs_from_beta  = 2*beta**2 * a              # independent route; must agree
    return dict(beta=beta, kappa=kappa, closure_residual=closure_residual,
                a=a, Rs_from_kappa=Rs_from_kappa, Rs_from_beta=Rs_from_beta,
                rp=a*(1-e), ra=a*(1+e))

def rom_phase(beta, e, a, Rs, O):
    """Phase-resolved state from ROM, using only observation-derived a, Rs."""
    r       = a*(1-e**2)/(1+e*cos(O))                       # = Rs/kappa_o^2
    kappa_o = sqrt(Rs/r)
    beta_o  = sqrt(Rs/r - Rs/(2*a))                         # vis-viva, ROM form
    beta_T  = beta*(1+e*cos(O))/sqrt(1-e**2)
    beta_R  = beta*e*sin(O)/sqrt(1-e**2)
    zk = 1/sqrt(1-kappa_o**2) - 1
    zb = 1/sqrt(1-beta_o**2)  - 1
    return dict(r=r, beta_o=beta_o, beta_T=beta_T, beta_R=beta_R,
                kappa_o=kappa_o, z_kappa=zk, z_beta=zb, Zsys=(1+zk)*(1+zb))

# ======================================================================
# 3.  RUN CLOSURE FROM OBSERVABLES + VERIFY against truth
# ======================================================================
def rel(x, y):
    return abs(x-y)/max(abs(y), mp.mpf(1))

print("="*78)
print("R.O.M.  MASS/G-FREE CLOSURE")
print("="*78)
print("Truth (hidden from closure): M = 1.4 Msun, a = %.6e m, e = %.2f" % (float(a_true), float(e)))
print("Observable inputs handed to ROM: beta=%.6f, e=%.2f, T=%.6f s" % (float(OBS['beta']), float(OBS['e']), float(OBS['T'])))
print("                                 z_kappa=%.6e, z_beta=%.6e" % (float(OBS['z_kappa']), float(OBS['z_beta'])))

R = rom_from_kinematics(OBS['beta'], OBS['e'], OBS['T'], c)

print("\n--- Geometry recovered from {beta, e, T} alone (NO G, NO M) ---")
rows = [("a   (semi-major axis)", R['a'], a_true),
        ("R_s (Schwarzschild radius)", R['Rs'], Rs_true),
        ("r_p (periapsis)", R['rp'], rp_true),
        ("r_a (apoapsis)", R['ra'], ra_true),
        ("kappa", R['kappa'], kappa_tru)]
allok = True
for name, got, tru in rows:
    ok = rel(got, tru) < mp.mpf('1e-25'); allok &= ok
    print("   %-26s ROM=%.10e  truth=%.10e  rel=%.1e  %s"
          % (name, float(got), float(tru), float(rel(got, tru)), "OK" if ok else "**FAIL**"))

print("\n--- Mass parameter as an OUTPUT (this is where G, M finally appear) ---")
GM_true = G*M_true
ok_gm = rel(R['GM'], GM_true) < mp.mpf('1e-25'); allok &= ok_gm
print("   GM = beta^2 c^2 a     ROM=%.10e  truth=%.10e  rel=%.1e  %s   (NO G used)"
      % (float(R['GM']), float(GM_true), float(rel(R['GM'], GM_true)), "OK" if ok_gm else "**FAIL**"))
M_kg = R['Rs']*c**2/(2*G)          # <-- the ONLY place G is used: length -> kg
ok_m = rel(M_kg, M_true) < mp.mpf('1e-25'); allok &= ok_m
print("   M  = R_s c^2 /(2G)    ROM=%.10e  truth=%.10e  rel=%.1e  %s   (G = unit conversion)"
      % (float(M_kg), float(M_true), float(rel(M_kg, M_true)), "OK" if ok_m else "**FAIL**"))
print("   M  in solar masses:   ROM=%.8f  truth=%.8f" % (float(M_kg/Msun), float(M_true/Msun)))

# ----- contrast with the classical route -----
print("\n--- Why this is non-trivial: classical scale needs GM, ROM's does not ---")
a_classical = (G*M_true*T_true**2/(4*pi**2))**(mp.mpf(1)/3)   # Kepler: requires GM as INPUT
print("   classical a = (G M T^2/4pi^2)^(1/3) = %.6e m   <- REQUIRES GM as input" % float(a_classical))
print("   ROM      a = T beta c /(2 pi)       = %.6e m   <- requires only T, beta" % float(R['a']))

# ======================================================================
# 4.  Redshift-mode closure + on-data closure test
# ======================================================================
print("\n" + "="*78)
print("CLOSURE FROM REDSHIFTS {z_kappa, z_beta, e, T}  (+ on-data closure test)")
print("="*78)
Rz = rom_from_redshifts(OBS['z_kappa'], OBS['z_beta'], OBS['e'], OBS['T'], c)
print("   beta  recovered = %.12f   (truth %.12f)" % (float(Rz['beta']), float(beta_glob)))
print("   kappa recovered = %.12f   (truth %.12f)" % (float(Rz['kappa']), float(kappa_tru)))
print("   closure test  kappa^2 - 2 beta^2 = %.2e   (expect 0 if data obeys ROM)" % float(Rz['closure_residual']))
print("   R_s via kappa = %.10e" % float(Rz['Rs_from_kappa']))
print("   R_s via beta  = %.10e   (two independent routes agree: %s)"
      % (float(Rz['Rs_from_beta']),
         rel(Rz['Rs_from_kappa'], Rz['Rs_from_beta']) < mp.mpf('1e-25')))

# ======================================================================
# 5.  Input-symmetry: several different observable sets -> same system
# ======================================================================
print("\n" + "="*78)
print("INPUT-SYMMETRY: different observable sets close to the SAME geometry")
print("="*78)
# Set A already done: {beta, e, T}
# Set C: {R_s, beta, e}  (someone measured R_s and beta)   -> a, T
a_C  = R['Rs']/(2*OBS['beta']**2)
T_C  = 2*pi*a_C/(OBS['beta']*c)
# Set D: {a, e, T}  -> beta, Rs
beta_D = 2*pi*R['a']/(T_true*c)
Rs_D   = 2*beta_D**2*R['a']
print("   Set A {beta,e,T} : a=%.6e  Rs=%.6e" % (float(R['a']), float(R['Rs'])))
print("   Set C {Rs,beta,e}: a=%.6e  T=%.6f s  (T matches: %s)"
      % (float(a_C), float(T_C), rel(T_C, T_true) < mp.mpf('1e-25')))
print("   Set D {a,e,T}    : beta=%.8f  Rs=%.6e  (beta,Rs match: %s)"
      % (float(beta_D), float(Rs_D),
         (rel(beta_D, beta_glob) < mp.mpf('1e-25') and rel(Rs_D, R['Rs']) < mp.mpf('1e-25'))))

# ======================================================================
# 6.  Phase-resolved orbit reconstructed from observables vs truth
# ======================================================================
print("\n" + "="*78)
print("PHASE-RESOLVED ORBIT  (ROM from observables)  vs  TRUTH (standard physics)")
print("="*78)
print("   O[deg]     r/a        beta_o(ROM/tru)        z_beta(ROM/tru)     ok")
phase_ok = True
for deg in [0, 30, 60, 90, 150, 210, 300]:
    O = mp.mpf(deg)*pi/180
    P = rom_phase(OBS['beta'], OBS['e'], R['a'], R['Rs'], O)
    Tr = truth_phase(O)
    ok = (rel(P['beta_o'], Tr['beta_o']) < mp.mpf('1e-22')
          and rel(P['z_beta'], Tr['z_beta']) < mp.mpf('1e-22')
          and rel(P['r'], Tr['r']) < mp.mpf('1e-22')
          and rel(P['kappa_o'], Tr['kappa_o']) < mp.mpf('1e-22'))
    phase_ok &= ok
    print("   %5d   %8.5f   %.10f/%.10f   %.4e/%.4e  %s"
          % (deg, float(P['r']/R['a']), float(P['beta_o']), float(Tr['beta_o']),
             float(P['z_beta']), float(Tr['z_beta']), "OK" if ok else "FAIL"))

# ======================================================================
# 7.  Ledger
# ======================================================================
print("\n" + "="*78)
print("LEDGER")
print("="*78)
print("  Inputs actually used by closure : beta (or z_beta), kappa (or z_kappa), e, T, c")
print("  Quantities that are OUTPUTS     : a, R_s, r_p, r_a, full orbit, GM, and M")
print("  Where G appears                 : ONLY in M = R_s c^2/(2G)  (length -> kg)")
print("  Where M appears                 : ONLY as that final output")
print("  GM itself                       : = beta^2 c^2 a  (no G needed at all)")
print("  Differential equations used     : NONE")
print("  Metric / curvature / geodesic   : NONE")
print("\n  OVERALL: %s" % ("ALL CHECKS PASS" if (allok and phase_ok) else "SOME CHECKS FAILED"))

R.O.M.  MASS/G-FREE CLOSURE
Truth (hidden from closure): M = 1.4 Msun, a = 2.067805e+07 m, e = 0.88
Observable inputs handed to ROM: beta=0.010000, e=0.88, T=43.337997 s
                                 z_kappa=1.000150e-04, z_beta=5.000375e-05

--- Geometry recovered from {beta, e, T} alone (NO G, NO M) ---
   a   (semi-major axis)      ROM=2.0678054155e+07  truth=2.0678054155e+07  rel=1.6e-31  OK
   R_s (Schwarzschild radius) ROM=4.1356108311e+03  truth=4.1356108311e+03  rel=0.0e+00  OK
   r_p (periapsis)            ROM=2.4813664987e+06  truth=2.4813664987e+06  rel=1.7e-31  OK
   r_a (apoapsis)             ROM=3.8874741812e+07  truth=3.8874741812e+07  rel=1.7e-31  OK
   kappa                      ROM=1.4142135624e-02  truth=1.4142135624e-02  rel=0.0e+00  OK

--- Mass parameter as an OUTPUT (this is where G, M finally appear) ---
   GM = beta^2 c^2 a     ROM=1.8584508258e+20  truth=1.8584508258e+20  rel=1.6e-31  OK   (NO G used)
   M  = R_s c^2 /(2G)    ROM=2.7844880000e+30  truth=2.7